# Searching Algorithms

A comprehensive guide to searching algorithms with deep focus on Binary Search.

---

## Linear Search

Check each element one by one. Works on **unsorted** data.

### Algorithm
- Iterate through array from start to end
- Compare each element with target
- Return index when found, -1 if not found

### Complexity
- **Time**: O(n) - must check every element in worst case
- **Space**: O(1) - no extra space needed

In [ ]:
def linear_search(arr, target):
    """O(n) time, O(1) space"""
    for i, val in enumerate(arr):
        if val == target:
            return i
    return -1

arr = [64, 34, 25, 12, 22, 11, 90]
print(f"Array: {arr}")
print(f"Index of 22: {linear_search(arr, 22)}")
print(f"Index of 100: {linear_search(arr, 100)}")

---

## Binary Search - Deep Dive

### High-Level Algorithm

Binary Search is a **divide-and-conquer** algorithm that efficiently searches **sorted** arrays by repeatedly dividing the search space in half.

#### Core Idea
```
1. Start with entire array (left=0, right=n-1)
2. Find middle element
3. Compare middle with target:
   - If equal: Found! Return index
   - If middle < target: Search right half
   - If middle > target: Search left half
4. Repeat until found or search space exhausted
```

#### Why O(log n)?
Each comparison eliminates **half** of remaining elements:
```
n elements → n/2 → n/4 → n/8 → ... → 1
Number of steps = log₂(n)
```

#### Visual Example
```
Array: [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
Target: 7

Step 1: left=0, right=9, mid=4
        [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
                     ^
        arr[4]=9 > 7, search left half

Step 2: left=0, right=3, mid=1
        [1, 3, 5, 7]
            ^
        arr[1]=3 < 7, search right half

Step 3: left=2, right=3, mid=2
        [5, 7]
         ^
        arr[2]=5 < 7, search right half

Step 4: left=3, right=3, mid=3
        [7]
         ^
        arr[3]=7 == 7, FOUND at index 3!
```

---

## Binary Search - Critical Implementation Details

### Why `mid = left + (right - left) // 2`?

**Problem with `mid = (left + right) // 2`**:
- In languages like C/Java: `left + right` can **overflow** if both are large
- Example: `left = 2^30`, `right = 2^30` → `left + right = 2^31` (overflow in 32-bit int)

**Solution: `mid = left + (right - left) // 2`**:
- Mathematically equivalent: `left + (right - left) // 2 = (2*left + right - left) // 2 = (left + right) // 2`
- **Never overflows**: `(right - left)` is always smaller than `right`
- **Best practice** even in Python (no overflow) for consistency

```python
# Both are equivalent in Python:
mid = (left + right) // 2           # Simple, works in Python
mid = left + (right - left) // 2    # Overflow-safe, universal
```

### Loop Condition: `while left <= right`

**Why `<=` and not `<`?**

```
Scenario: Array = [5], Target = 5

Initial: left=0, right=0

With left < right:
  - Loop doesn't execute (0 < 0 is False)
  - Returns -1 (WRONG! Element exists)

With left <= right:
  - Loop executes (0 <= 0 is True)
  - mid = 0, arr[0] = 5 = target
  - Returns 0 (CORRECT!)
```

**Rule**: Use `left <= right` to include the case where search space has exactly one element.

---

## Loop Termination Analysis

### How Does the Loop Exit?

The loop `while left <= right` exits when `left > right`.

#### Case 1: Target Found
```python
if arr[mid] == target:
    return mid  # Exit immediately
```

#### Case 2: Target Not Found - Search Space Exhausted

Let's trace the final steps:

```
Array: [1, 3, 5, 7, 9]
Target: 6 (doesn't exist)

...(earlier steps)...

Step N-2: left=2, right=3
          [5, 7]
          mid = 2 + (3-2)//2 = 2
          arr[2]=5 < 6, so left = mid + 1 = 3

Step N-1: left=3, right=3  ← Last iteration
          [7]
          mid = 3 + (3-3)//2 = 3
          arr[3]=7 > 6, so right = mid - 1 = 2

Step N: left=3, right=2
        left > right (3 > 2) ← Loop exits!
        Return -1 (not found)
```

#### Key Insight: Final State When Not Found

When loop exits with `left > right`:
- `left` points to where target **should be inserted**
- `right` points to the largest element **smaller than target**
- `left = right + 1` always

```
Example: Insert 6 into [1, 3, 5, 7, 9]

Final state: left=3, right=2
             [1, 3, 5, 7, 9]
                    ↑ ↑
                right left

Insert at index 3: [1, 3, 5, 6, 7, 9]
                              ↑
                           left=3
```

---

## Standard Binary Search Implementation

In [ ]:
def binary_search(arr, target):
    """
    Standard binary search.
    Returns index if found, -1 otherwise.
    
    Time: O(log n)
    Space: O(1)
    """
    left, right = 0, len(arr) - 1
    
    while left <= right:
        # Overflow-safe mid calculation
        mid = left + (right - left) // 2
        
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1  # Search right half
        else:
            right = mid - 1  # Search left half
    
    return -1  # Not found

# Test
arr = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
print(f"Array: {arr}\n")
print(f"Search for 7: index {binary_search(arr, 7)}")
print(f"Search for 15: index {binary_search(arr, 15)}")
print(f"Search for 6 (not exists): {binary_search(arr, 6)}")
print(f"Search for 20 (not exists): {binary_search(arr, 20)}")

## Binary Search with Detailed Trace

In [ ]:
def binary_search_verbose(arr, target):
    """
    Binary search with step-by-step trace.
    Shows how left, right, and mid change in each iteration.
    """
    left, right = 0, len(arr) - 1
    step = 0
    
    print(f"Searching for {target} in {arr}\n")
    print(f"{'Step':<6} {'left':<6} {'right':<6} {'mid':<6} {'arr[mid]':<10} {'Action'}")
    print("="*60)
    
    while left <= right:
        step += 1
        mid = left + (right - left) // 2
        
        print(f"{step:<6} {left:<6} {right:<6} {mid:<6} {arr[mid]:<10}", end=" ")
        
        if arr[mid] == target:
            print(f"FOUND at index {mid}!")
            return mid
        elif arr[mid] < target:
            print(f"{arr[mid]} < {target}, search right")
            left = mid + 1
        else:
            print(f"{arr[mid]} > {target}, search left")
            right = mid - 1
    
    print(f"\nLoop exits: left={left}, right={right} (left > right)")
    print(f"Target {target} not found. Would insert at index {left}")
    return -1

# Test cases
arr = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]

print("Example 1: Target exists\n")
binary_search_verbose(arr, 7)

print("\n" + "="*60 + "\n")
print("Example 2: Target doesn't exist (middle)\n")
binary_search_verbose(arr, 6)

print("\n" + "="*60 + "\n")
print("Example 3: Target doesn't exist (before first)\n")
binary_search_verbose(arr, 0)

print("\n" + "="*60 + "\n")
print("Example 4: Target doesn't exist (after last)\n")
binary_search_verbose(arr, 20)

---

## Binary Search: Insert Position

### Problem
Given a sorted array and a target, return the index where target should be inserted to maintain sorted order.

### Key Insight
When binary search doesn't find the target, `left` points to the **insertion position**!

```
Array: [1, 3, 5, 7, 9]

Insert 6:
  Final: left=3, right=2
  Insert at index 3: [1, 3, 5, 6, 7, 9]

Insert 0:
  Final: left=0, right=-1
  Insert at index 0: [0, 1, 3, 5, 7, 9]

Insert 10:
  Final: left=5, right=4
  Insert at index 5: [1, 3, 5, 7, 9, 10]
```

In [ ]:
def search_insert_position(arr, target):
    """
    Find index where target should be inserted.
    If target exists, return its index.
    If not, return where it should be inserted.
    
    LeetCode 35: Search Insert Position
    """
    left, right = 0, len(arr) - 1
    
    while left <= right:
        mid = left + (right - left) // 2
        
        if arr[mid] == target:
            return mid  # Found exact match
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    
    # When loop exits: left > right
    # left is the insertion position
    return left

# Test
arr = [1, 3, 5, 7, 9]
print(f"Array: {arr}\n")

test_cases = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
for target in test_cases:
    pos = search_insert_position(arr, target)
    if pos < len(arr) and arr[pos] == target:
        print(f"Target {target}: Found at index {pos}")
    else:
        print(f"Target {target}: Insert at index {pos}")

## Insert Position with Detailed Trace

In [ ]:
def search_insert_verbose(arr, target):
    """
    Insert position with detailed trace showing final left/right state.
    """
    left, right = 0, len(arr) - 1
    step = 0
    
    print(f"Finding insert position for {target} in {arr}\n")
    print(f"{'Step':<6} {'left':<6} {'right':<6} {'mid':<6} {'arr[mid]':<10} {'Action'}")
    print("="*70)
    
    while left <= right:
        step += 1
        mid = left + (right - left) // 2
        
        print(f"{step:<6} {left:<6} {right:<6} {mid:<6} {arr[mid]:<10}", end=" ")
        
        if arr[mid] == target:
            print(f"FOUND at index {mid}")
            print(f"\nResult: Target exists at index {mid}")
            return mid
        elif arr[mid] < target:
            print(f"{arr[mid]} < {target}, search right")
            left = mid + 1
        else:
            print(f"{arr[mid]} > {target}, search left")
            right = mid - 1
    
    print(f"\nLoop exits: left={left}, right={right}")
    print(f"Condition: left > right ({left} > {right})")
    print(f"\nInsertion position: {left}")
    
    # Show what array would look like after insertion
    new_arr = arr[:left] + [target] + arr[left:]
    print(f"After insertion: {new_arr}")
    
    return left

# Examples
arr = [1, 3, 5, 7, 9]

print("Example 1: Insert in middle\n")
search_insert_verbose(arr, 6)

print("\n" + "="*70 + "\n")
print("Example 2: Insert at beginning\n")
search_insert_verbose(arr, 0)

print("\n" + "="*70 + "\n")
print("Example 3: Insert at end\n")
search_insert_verbose(arr, 10)

print("\n" + "="*70 + "\n")
print("Example 4: Target exists\n")
search_insert_verbose(arr, 5)

---

## Edge Cases and Special Scenarios

### Edge Case 1: Empty Array

In [ ]:
print("Edge Case 1: Empty Array\n")
arr = []
print(f"Array: {arr}")
print(f"Search for 5: {binary_search(arr, 5)}")
print(f"Insert position for 5: {search_insert_position(arr, 5)}")
print("\nExplanation:")
print("  Initial: left=0, right=-1")
print("  Condition: left <= right → 0 <= -1 → False")
print("  Loop doesn't execute, returns left=0")

### Edge Case 2: Single Element Array

In [ ]:
print("Edge Case 2: Single Element Array\n")
arr = [5]
print(f"Array: {arr}\n")

# Target equals element
print("Scenario A: Target = 5 (equals element)")
print(f"  Search: {binary_search(arr, 5)}")
print(f"  Insert: {search_insert_position(arr, 5)}")
print("  Trace: left=0, right=0, mid=0, arr[0]=5==5, return 0\n")

# Target less than element
print("Scenario B: Target = 3 (less than element)")
print(f"  Search: {binary_search(arr, 3)}")
print(f"  Insert: {search_insert_position(arr, 3)}")
print("  Trace: left=0, right=0, mid=0, arr[0]=5>3")
print("         right=mid-1=-1, left=0, right=-1")
print("         left > right, return left=0\n")

# Target greater than element
print("Scenario C: Target = 7 (greater than element)")
print(f"  Search: {binary_search(arr, 7)}")
print(f"  Insert: {search_insert_position(arr, 7)}")
print("  Trace: left=0, right=0, mid=0, arr[0]=5<7")
print("         left=mid+1=1, left=1, right=0")
print("         left > right, return left=1")

### Edge Case 3: Two Element Array

In [ ]:
print("Edge Case 3: Two Element Array\n")
arr = [3, 7]
print(f"Array: {arr}\n")

for target in [1, 3, 5, 7, 9]:
    result = search_insert_position(arr, target)
    print(f"Target {target}: position {result}")

### Edge Case 4: All Elements Same

In [ ]:
print("Edge Case 4: All Elements Same\n")
arr = [5, 5, 5, 5, 5]
print(f"Array: {arr}\n")

print(f"Search for 5: {binary_search(arr, 5)}")
print(f"Insert position for 5: {search_insert_position(arr, 5)}")
print(f"Search for 3: {binary_search(arr, 3)}")
print(f"Insert position for 3: {search_insert_position(arr, 3)}")
print(f"Search for 7: {binary_search(arr, 7)}")
print(f"Insert position for 7: {search_insert_position(arr, 7)}")

---

## Binary Search Variants

### Find First Occurrence

When array has duplicates, find the **leftmost** (first) occurrence.

In [ ]:
def find_first_occurrence(arr, target):
    """
    Find first (leftmost) occurrence of target.
    Returns -1 if not found.
    """
    left, right = 0, len(arr) - 1
    result = -1
    
    while left <= right:
        mid = left + (right - left) // 2
        
        if arr[mid] == target:
            result = mid      # Found, but keep searching left
            right = mid - 1   # Continue searching in left half
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    
    return result

# Test
arr = [1, 2, 2, 2, 2, 3, 4, 5]
print(f"Array: {arr}")
print(f"First occurrence of 2: index {find_first_occurrence(arr, 2)}")
print(f"First occurrence of 5: index {find_first_occurrence(arr, 5)}")
print(f"First occurrence of 6: {find_first_occurrence(arr, 6)}")

### Find Last Occurrence

Find the **rightmost** (last) occurrence.

In [ ]:
def find_last_occurrence(arr, target):
    """
    Find last (rightmost) occurrence of target.
    Returns -1 if not found.
    """
    left, right = 0, len(arr) - 1
    result = -1
    
    while left <= right:
        mid = left + (right - left) // 2
        
        if arr[mid] == target:
            result = mid      # Found, but keep searching right
            left = mid + 1    # Continue searching in right half
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    
    return result

# Test
arr = [1, 2, 2, 2, 2, 3, 4, 5]
print(f"Array: {arr}")
print(f"Last occurrence of 2: index {find_last_occurrence(arr, 2)}")
print(f"Last occurrence of 5: index {find_last_occurrence(arr, 5)}")
print(f"Last occurrence of 6: {find_last_occurrence(arr, 6)}")

### Count Occurrences

Using first and last occurrence:

In [ ]:
def count_occurrences(arr, target):
    """
    Count occurrences of target in O(log n) time.
    """
    first = find_first_occurrence(arr, target)
    if first == -1:
        return 0
    
    last = find_last_occurrence(arr, target)
    return last - first + 1

# Test
arr = [1, 2, 2, 2, 2, 3, 4, 5]
print(f"Array: {arr}")
print(f"Count of 2: {count_occurrences(arr, 2)}")
print(f"Count of 1: {count_occurrences(arr, 1)}")
print(f"Count of 6: {count_occurrences(arr, 6)}")

---

## Binary Search on Answer Space

### Concept

Sometimes we don't search in an array, but search for an **answer** in a range.

**Pattern**: If we can verify whether a value works in O(f(n)), we can binary search for the optimal value in O(f(n) * log(range)).

### Example: Integer Square Root

In [ ]:
def sqrt_integer(n):
    """
    Find integer square root using binary search.
    Returns largest integer k where k² <= n.
    
    Example: sqrt(16) = 4, sqrt(17) = 4
    """
    if n < 2:
        return n
    
    left, right = 1, n // 2
    result = 0
    
    while left <= right:
        mid = left + (right - left) // 2
        square = mid * mid
        
        if square == n:
            return mid
        elif square < n:
            result = mid      # mid might be answer, save it
            left = mid + 1    # Try larger values
        else:
            right = mid - 1   # mid is too large
    
    return result

# Test
test_values = [0, 1, 4, 8, 16, 17, 24, 25, 100, 101]
print("Integer Square Root:\n")
for n in test_values:
    result = sqrt_integer(n)
    print(f"sqrt({n:3}) = {result:2}  (verify: {result}² = {result*result})")

### Example: Find Peak Element

A peak element is greater than its neighbors.

In [ ]:
def find_peak_element(arr):
    """
    Find any peak element in O(log n).
    Peak: arr[i] > arr[i-1] and arr[i] > arr[i+1]
    
    Key insight: Always move toward higher neighbor.
    """
    left, right = 0, len(arr) - 1
    
    while left < right:
        mid = left + (right - left) // 2
        
        if arr[mid] > arr[mid + 1]:
            # Peak is on left side (including mid)
            right = mid
        else:
            # Peak is on right side
            left = mid + 1
    
    return left  # left == right at peak

# Test
test_arrays = [
    [1, 2, 3, 1],
    [1, 2, 1, 3, 5, 6, 4],
    [1, 2, 3, 4, 5],
    [5, 4, 3, 2, 1]
]

print("Find Peak Element:\n")
for arr in test_arrays:
    peak_idx = find_peak_element(arr)
    print(f"Array: {arr}")
    print(f"Peak at index {peak_idx}, value = {arr[peak_idx]}\n")

---

## Python's bisect Module

Python provides built-in binary search via the `bisect` module.

In [ ]:
import bisect

arr = [1, 3, 4, 4, 4, 6, 8]
print(f"Array: {arr}\n")

# bisect_left: leftmost insertion point
print("bisect_left (insert before existing):")
print(f"  bisect_left(arr, 4) = {bisect.bisect_left(arr, 4)}  # Insert before first 4")
print(f"  bisect_left(arr, 5) = {bisect.bisect_left(arr, 5)}  # Insert at position 5")
print(f"  bisect_left(arr, 0) = {bisect.bisect_left(arr, 0)}  # Insert at start")
print(f"  bisect_left(arr, 9) = {bisect.bisect_left(arr, 9)}  # Insert at end\n")

# bisect_right (or bisect): rightmost insertion point
print("bisect_right (insert after existing):")
print(f"  bisect_right(arr, 4) = {bisect.bisect_right(arr, 4)}  # Insert after last 4")
print(f"  bisect_right(arr, 5) = {bisect.bisect_right(arr, 5)}  # Insert at position 5")
print(f"  bisect_right(arr, 0) = {bisect.bisect_right(arr, 0)}  # Insert at start")
print(f"  bisect_right(arr, 9) = {bisect.bisect_right(arr, 9)}  # Insert at end\n")

# insort: insert and maintain sorted order
arr_copy = arr.copy()
bisect.insort(arr_copy, 5)
print(f"After insort(5): {arr_copy}")

# Using bisect for range queries
print("\nCount elements in range [4, 6]:")
left_idx = bisect.bisect_left(arr, 4)
right_idx = bisect.bisect_right(arr, 6)
print(f"  Elements from index {left_idx} to {right_idx-1}")
print(f"  Count: {right_idx - left_idx}")
print(f"  Elements: {arr[left_idx:right_idx]}")

---

## Recursive Binary Search

In [ ]:
def binary_search_recursive(arr, target, left=0, right=None):
    """
    Recursive binary search.
    Time: O(log n)
    Space: O(log n) due to recursion stack
    """
    if right is None:
        right = len(arr) - 1
    
    # Base case: search space exhausted
    if left > right:
        return -1
    
    mid = left + (right - left) // 2
    
    if arr[mid] == target:
        return mid
    elif arr[mid] < target:
        return binary_search_recursive(arr, target, mid + 1, right)
    else:
        return binary_search_recursive(arr, target, left, mid - 1)

# Test
arr = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
print(f"Array: {arr}\n")
print(f"Recursive search for 7: {binary_search_recursive(arr, 7)}")
print(f"Recursive search for 15: {binary_search_recursive(arr, 15)}")
print(f"Recursive search for 6: {binary_search_recursive(arr, 6)}")

---

## Comparison of Search Algorithms

| Algorithm | Time | Space | Requires Sorted | Best For |
|-----------|------|-------|----------------|----------|
| **Linear Search** | O(n) | O(1) | ❌ No | Small arrays, unsorted data |
| **Binary Search** | O(log n) | O(1) | ✅ Yes | Large sorted arrays |
| **Binary Search (Recursive)** | O(log n) | O(log n) | ✅ Yes | Educational, cleaner code |
| **Hash Table** | O(1) avg | O(n) | ❌ No | Frequent lookups, extra space OK |

---

## Binary Search Template

### Standard Template

```python
def binary_search_template(arr, target):
    left, right = 0, len(arr) - 1
    
    while left <= right:
        mid = left + (right - left) // 2
        
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    
    return -1  # or return left for insert position
```

### Key Points to Remember

1. **Initialization**: `left = 0, right = len(arr) - 1`
2. **Loop condition**: `while left <= right` (include equality!)
3. **Mid calculation**: `mid = left + (right - left) // 2` (overflow-safe)
4. **Update pointers**: 
   - `left = mid + 1` (exclude mid, search right)
   - `right = mid - 1` (exclude mid, search left)
5. **Exit condition**: `left > right`
6. **Insert position**: When not found, `left` is the insertion index

---

## Common Mistakes and How to Avoid Them

### Mistake 1: Using `left < right` instead of `left <= right`

```python
# WRONG: Misses single-element case
while left < right:
    ...

# CORRECT
while left <= right:
    ...
```

### Mistake 2: Integer overflow in mid calculation

```python
# RISKY: Can overflow in C/Java
mid = (left + right) // 2

# SAFE: Always use this
mid = left + (right - left) // 2
```

### Mistake 3: Not excluding mid when updating

```python
# WRONG: Infinite loop possible
if arr[mid] < target:
    left = mid  # Should be mid + 1

# CORRECT
if arr[mid] < target:
    left = mid + 1
```

### Mistake 4: Forgetting array must be sorted

```python
# WRONG: Binary search on unsorted array
arr = [5, 2, 8, 1, 9]
binary_search(arr, 8)  # May return wrong result!

# CORRECT: Sort first
arr.sort()
binary_search(arr, 8)
```

---

## Key Takeaways

### Binary Search Fundamentals

1. **Requires sorted data** - won't work on unsorted arrays
2. **O(log n) time** - eliminates half of search space each iteration
3. **O(1) space** - iterative version uses constant space

### Critical Implementation Details

1. **Mid calculation**: Use `mid = left + (right - left) // 2` to avoid overflow
2. **Loop condition**: Use `left <= right` to handle single-element case
3. **Pointer updates**: Always exclude mid (`mid + 1` or `mid - 1`)
4. **Exit state**: When `left > right`, `left` points to insertion position

### Loop Termination

1. **Found**: Return immediately when `arr[mid] == target`
2. **Not found**: Loop exits when `left > right`
3. **Final state**: `left = right + 1`
4. **Insert position**: `left` is where target should be inserted

### Variants and Applications

1. **Find first/last occurrence** - modify to continue searching after finding
2. **Insert position** - return `left` when not found
3. **Search on answer** - binary search on value range, not array
4. **Python bisect** - use built-in module for production code

### When to Use Binary Search

✅ **Use when**:
- Data is sorted (or can be sorted)
- Need O(log n) search time
- Searching for value or insertion point
- Searching on monotonic answer space

❌ **Don't use when**:
- Data is unsorted and can't be sorted
- Array is very small (linear search is simpler)
- Need to find all occurrences (use linear or hash table)

### Remember

🎯 **Sorted data required**  
🎯 **`left <= right` loop condition**  
🎯 **`mid = left + (right - left) // 2`**  
🎯 **Exclude mid when updating (`mid ± 1`)**  
🎯 **`left` is insertion position when not found**  
🎯 **O(log n) time, O(1) space**  

Master binary search and you'll have one of the most powerful algorithmic tools!